# LoRaWAN: sensors kilometres away

A frame on the air, an over-the-air join by hand, the airtime budget, and a whole network — gateways speaking the Semtech UDP protocol to a light network server — running in memory.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.17.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.17.0-preview.1"

## A frame on the air
The classic example: 17 bytes, the payload is encrypted with the AppSKey and signed with the NwkSKey.

In [ ]:
using IoTCom.Net.Protocols.LoRaWan;

var phy = Convert.FromHexString("40F17DBE4900020001954378762B11FF0D");
foreach (var f in LoRaWanAnatomy.Describe(phy)) Console.WriteLine($"{f.Name,-11} {Convert.ToHexString(phy, f.Offset, f.Length),-10} {f.Value}");
var keys = LoRaWanSessionKeys.FromHex("44024241ED4CE9A68C6A8BC055233FD3", "EC925802AE430CA77FD3DD73CB2CC588");
var frame = LoRaWanPacket.Decode(phy);
Console.WriteLine($"MIC valid: {frame.VerifyMic(keys.NwkSKey)}, payload: {System.Text.Encoding.ASCII.GetString(frame.DecryptPayload(keys))}");

## Joining over the air, by hand
The device and the network derive the same session keys from the AppKey, the DevNonce and the Join-Accept. `LoRaWanEndDevice` is the device's MAC layer without any I/O.

In [ ]:
var appKey = LoRaWanKeys.Parse("2B7E151628AED2A6ABF7158809CF4F3C");
var node = new LoRaWanEndDevice(Eui64.Parse("70B3D57ED0000101"), default, appKey);
var joinRequest = LoRaWanPacket.Decode(node.CreateJoinRequest());
Console.WriteLine(joinRequest);
var accept = new LoRaWanJoinAccept(JoinNonce: 0x00A1B2, NetId: 0x13, DevAddr: DevAddr.Parse("26011BDA"));
node.HandleDownlink(accept.Encode(appKey));
var networkKeys = accept.DeriveSessionKeys(appKey, joinRequest.DevNonce);
Console.WriteLine($"joined as {node.DevAddr}; same keys on both sides: {node.SessionKeys!.NwkSKey.SequenceEqual(networkKeys.NwkSKey)}");
var uplink = LoRaWanPacket.Decode(node.CreateUplink(1, new CayenneLpp().AddTemperature(1, 28.4).AddHumidity(2, 71).ToArray()));
Console.WriteLine($"{uplink} → {string.Join(", ", CayenneLpp.Decode(uplink.DecryptPayload(networkKeys)))}");

## The airtime budget
Every step from SF7 to SF12 doubles the time on air: more range, less capacity. In EU868 a 1 % duty cycle turns that into a mandatory wait.

In [ ]:
foreach (var sf in new[] { 7, 8, 9, 10, 11, 12 })
{
    var air = LoRaAirtime.Compute(12 + 13, sf);
    Console.WriteLine($"SF{sf,-2} {air.TotalMilliseconds,7:0.0} ms   then wait {air.TotalSeconds * 99,6:0.0} s at 1 %");
}

## A network in memory
Two gateways forward over Semtech UDP to the network server; four sensors join and report. `HonorTimestamps = false` skips the real 5 s join delay so the cell finishes quickly.

In [ ]:
using System.Net;
using IoTCom.Net.Transports;

var radio = new InMemoryDatagramNetwork();
var nsAddress = new IPEndPoint(IPAddress.Parse("10.0.0.1"), 1700);
await using var ns = LoRaWanNetworkServer.Create(o => o.UseInMemory(radio, nsAddress).Region = LoRaRegion.AS923Group2);
var simOptions = LoRaWanSimulatorOptions.Demo(nsAddress);
simOptions.GatewayTransportFactory = () => radio.Bind();
simOptions.HonorTimestamps = false;
await using var sim = new LoRaWanSimulator(simOptions);
foreach (var r in sim.Registrations) ns.AddDevice(r);

var reports = new List<LoRaWanUplink>();
ns.UplinkReceived += (_, u) => { lock (reports) reports.Add(u); };
await ns.StartAsync();
await sim.StartAsync();
while (reports.Count < 4) await Task.Delay(100);
lock (reports)
    foreach (var u in reports)
        Console.WriteLine($"{u.Device.Name,-11} {u.DataRate,-9} via {u.Gateways.Count} gw, best SNR {u.Gateways[0].Snr,5:0.0}  {LoRaWanSimulator.DescribePayload(u.FPort, u.Payload)}");

## A downlink
Class A devices listen only right after they transmit, so a downlink waits in a queue for the next uplink. FPort 10 tells the simulated sensors how often to report.

In [ ]:
var weather = ns.Devices.First(d => d.Name == "weather-01");
var weatherNode = sim.Devices.First(d => d.Name == "weather-01");
ns.EnqueueDownlink(weather.DevEui, 10, [0x00, 0x3C]);
sim.TriggerUplink("weather-01");
while (weatherNode.Interval != TimeSpan.FromSeconds(60)) await Task.Delay(100);
Console.WriteLine($"weather-01 now reports every {weatherNode.Interval.TotalSeconds} s ({weather.DownlinkCount} downlinks)");

## Going further
The Gallery demo *LoRaWAN network monitor* puts gateways and sensors on a map, with every uplink and its RX windows. `iotcom lorawan server --sim` runs the same network in a terminal, and `iotcom lorawan simulate` drives ChirpStack or The Things Stack. See `docs/en/protocols/lorawan.md`.

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*